1. Setup: Imports & Tokenizer

In [1]:
import time
import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
from transformers import RobertaTokenizer, RobertaForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from datetime import timedelta
from tqdm.auto import tqdm
from torch.cuda.amp import autocast, GradScaler
import warnings
from tabulate import tabulate

warnings.filterwarnings("ignore")  # Suppress all Python warnings
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

2025-09-06 12:39:50.976885: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1757162391.000155     358 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1757162391.007408     358 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


2. Prepare Dataset

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load data
df = pd.read_csv("/kaggle/input/PURE_REQ.csv", sep=";")

# Map labels: Functional = 1, Non-Functional = 0
df["label"] = df["class"].map({"F": 1, "NFR": 0})

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["RequirementText"].tolist(), df["label"].tolist(), test_size=0.2, random_state=42
)

3. Tokenize Inputs

In [3]:
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

train_encodings = tokenizer(train_texts, truncation=True, padding=True, return_tensors="pt")
val_encodings = tokenizer(val_texts, truncation=True, padding=True, return_tensors="pt")


4. Dataset Class

In [4]:
class RequirementDataset(Dataset):
    def __init__(self, texts, labels):
        encodings = tokenizer(texts, truncation=True, padding=True, return_tensors='pt')
        self.encodings = encodings
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __getitem__(self, idx):
        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels': self.labels[idx]
        }
    def __len__(self):
        return len(self.labels)


6. Training Loop

In [5]:
# === Configuration Block ===
config = {
    "k_folds": 2,
    "epochs": 15,
    "batch_size": 128,
    "max_length": 128,
    "learning_rate": 1e-5,
    "use_amp": True,
    "gradient_accumulation_steps": 2,
    "model_name": "roberta-base",
    "num_labels": 2,
    "random_state": 42,
    "num_workers": 0,
    "pin_memory": True,
    "early_stopping_patience": 3
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
scaler = GradScaler()
skf = StratifiedKFold(n_splits=config["k_folds"], shuffle=True, random_state=config["random_state"])


In [6]:
# ==============================
# 📊 Cross-Validation Training
# ==============================

# Global storage
all_targets_global, all_preds_global = [], []
fold_f1_scores = []
all_folds_results = []

for fold, (train_idx, val_idx) in enumerate(skf.split(df["RequirementText"], df["label"])):
    print(f"\n🟦 Fold {fold+1}/{config['k_folds']} ==============================")

    # --- Prepare datasets ---
    train_texts = df.iloc[train_idx]["RequirementText"].tolist()
    train_labels = df.iloc[train_idx]["label"].tolist()
    val_texts = df.iloc[val_idx]["RequirementText"].tolist()
    val_labels = df.iloc[val_idx]["label"].tolist()

    def create_dataset(texts, labels):
        encodings = tokenizer(
            texts, truncation=True, padding=True, max_length=config["max_length"], return_tensors="pt"
        )
        class Dataset(torch.utils.data.Dataset):
            def __len__(self): return len(labels)
            def __getitem__(self, idx): return {
                "input_ids": encodings["input_ids"][idx],
                "attention_mask": encodings["attention_mask"][idx],
                "labels": torch.tensor(labels[idx], dtype=torch.long)
            }
        return Dataset()

    train_dataset = create_dataset(train_texts, train_labels)
    val_dataset = create_dataset(val_texts, val_labels)

    train_loader = DataLoader(train_dataset, batch_size=config["batch_size"], shuffle=True,
                              num_workers=config["num_workers"], pin_memory=config["pin_memory"])
    val_loader = DataLoader(val_dataset, batch_size=config["batch_size"],
                            num_workers=config["num_workers"], pin_memory=config["pin_memory"])

    # --- Model + Optimizer ---
    model = RobertaForSequenceClassification.from_pretrained(
        config["model_name"], num_labels=config["num_labels"]
    ).to(device)
    optimizer = AdamW(model.parameters(), lr=config["learning_rate"])

    best_f1, early_stop_counter = 0, 0
    best_preds, best_true = [], []

    # --- Epoch loop ---
    for epoch in range(config["epochs"]):
        start = time.time()

        # Training
        model.train()
        total_loss = 0
        for batch in tqdm(train_loader, desc=f"🟡 Fold {fold+1} | Epoch {epoch+1} [Train]", leave=False):
            batch = {k: v.to(device) for k, v in batch.items()}
            optimizer.zero_grad()
            with autocast(enabled=config["use_amp"]):
                outputs = model(**batch)
                loss = outputs.loss
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item()
        avg_train_loss = total_loss / len(train_loader)

        # Validation
        model.eval()
        val_loss, preds, true = 0, [], []
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"🔵 Fold {fold+1} | Epoch {epoch+1} [Eval]", leave=False):
                batch = {k: v.to(device) for k, v in batch.items()}
                with autocast(enabled=config["use_amp"]):
                    outputs = model(**batch)
                val_loss += outputs.loss.item()
                preds.extend(torch.argmax(outputs.logits, dim=1).cpu().numpy())
                true.extend(batch["labels"].cpu().numpy())
        avg_val_loss = val_loss / len(val_loader)
        val_f1 = f1_score(true, preds, average="binary")

        print(f"Epoch {epoch+1:02d} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | F1: {val_f1:.4f} | ⏱ {time.time()-start:.1f}s")

        # Early stopping
        if val_f1 > best_f1:
            best_f1, best_preds, best_true = val_f1, preds, true
            early_stop_counter = 0
        else:
            early_stop_counter += 1
            if early_stop_counter >= config["early_stopping_patience"]:
                print("🛑 Early stopping triggered.")
                break

    # --- Per-fold metrics ---
    precision = precision_score(best_true, best_preds, average="binary")
    recall = recall_score(best_true, best_preds, average="binary")
    f1 = f1_score(best_true, best_preds, average="binary")

    print(f"\n📊 Fold {fold+1} — Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}")
    print(classification_report(best_true, best_preds, target_names=["Non-Functional", "Functional"]))

    # Save results
    all_folds_results.append({
        "fold": fold+1,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })
    all_targets_global.extend(best_true)
    all_preds_global.extend(best_preds)
    fold_f1_scores.append(f1)

    # Free memory
    del model
    torch.cuda.empty_cache()
    import gc; gc.collect()




🟦 Fold 1/2 ==============================


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


🟡 Fold 1 | Epoch 1 [Train]:   0%|          | 0/17 [00:00<?, ?it/s]

🔵 Fold 1 | Epoch 1 [Eval]:   0%|          | 0/17 [00:00<?, ?it/s]

Epoch 01 | Train Loss: 0.6819 | Val Loss: 0.5666 | F1: 0.8588 | ⏱ 28.4s


...

🔵 Fold 2 | Epoch 14 [Eval]:   0%|          | 0/17 [00:00<?, ?it/s]

Epoch 14 | Train Loss: 0.0369 | Val Loss: 0.4761 | F1: 0.9089 | ⏱ 28.1s


🟡 Fold 2 | Epoch 15 [Train]:   0%|          | 0/17 [00:00<?, ?it/s]

🔵 Fold 2 | Epoch 15 [Eval]:   0%|          | 0/17 [00:00<?, ?it/s]

Epoch 15 | Train Loss: 0.0272 | Val Loss: 0.4554 | F1: 0.9262 | ⏱ 28.1s
🛑 Early stopping triggered.

📊 Fold 2 — Precision: 0.9048, Recall: 0.9506, F1: 0.9271
                precision    recall  f1-score   support

Non-Functional       0.82      0.70      0.75       513
    Functional       0.90      0.95      0.93      1559

      accuracy                           0.89      2072
     macro avg       0.86      0.82      0.84      2072
  weighted avg       0.88      0.89      0.88      2072



In [7]:
# ==============================
# 📊 Final Cross-Validation Results
# ==============================
final_df = pd.DataFrame(all_folds_results)
# print("\n✅ Final Results Across All Folds:")
# print(tabulate(final_df, headers="keys", tablefmt="fancy_grid"))
final_df.to_csv("training_results.csv", index=False)

# Aggregated Metrics (pooled across folds)
precision_macro = precision_score(all_targets_global, all_preds_global, average="macro", zero_division=0)
recall_macro = recall_score(all_targets_global, all_preds_global, average="macro", zero_division=0)
f1_macro = f1_score(all_targets_global, all_preds_global, average="macro", zero_division=0)

f1_mean = np.mean(fold_f1_scores)
f1_std = np.std(fold_f1_scores)

print("\n📊 Overall Classification Report (Aggregated across folds):")
print(
    classification_report(
        all_targets_global,
        all_preds_global,
        labels=[0, 1],
        target_names=["Non-Functional", "Functional"],
        zero_division=0
    )
)
print(f"\n📈 Fold-level F1 (mean ± std): {f1_mean:.4f} ± {f1_std:.4f}")
print(f"📌 Macro Precision: {precision_macro:.4f} | Macro Recall: {recall_macro:.4f} | Macro F1: {f1_macro:.4f}")



📊 Overall Classification Report (Aggregated across folds):
                precision    recall  f1-score   support

Non-Functional       0.82      0.70      0.76      1026
    Functional       0.91      0.95      0.93      3119

      accuracy                           0.89      4145
     macro avg       0.86      0.83      0.84      4145
  weighted avg       0.88      0.89      0.88      4145


📈 Fold-level F1 (mean ± std): 0.9268 ± 0.0003
📌 Macro Precision: 0.8615 | Macro Recall: 0.8254 | Macro F1: 0.8411


In [8]:
import shap


final_model = RobertaForSequenceClassification.from_pretrained(
    config["model_name"], num_labels=config["num_labels"]
).to(device)

optimizer = AdamW(final_model.parameters(), lr=config["learning_rate"])

# Prepare full dataset
full_dataset = create_dataset(df["RequirementText"].tolist(), df["label"].tolist())
full_loader = DataLoader(full_dataset, batch_size=config["batch_size"], shuffle=True)

# Train for a few epochs
final_model.train()
for epoch in range(config["epochs"]):
    for batch in full_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        outputs = final_model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [9]:
import shap
pd.set_option("display.max_colwidth", None)  # show full text in DataFrame

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ✅ Define correct prediction function
def predict(texts):
    enc = tokenizer(list(texts), truncation=True, padding=True,
                    max_length=config["max_length"], return_tensors="pt")
    enc = {k: v.to(device) for k, v in enc.items()}   # ✅ use the correct variable
    with torch.no_grad():
        outputs = final_model(**enc)
        probs = torch.softmax(outputs.logits, dim=1)[:, 1]  # Prob of Functional
        return probs.cpu().numpy()


try:
    final_model.eval()

    # Background & explainer
    background = df['RequirementText'].sample(20, random_state=42).tolist()
    masker = shap.maskers.Text(tokenizer)  # pass tokenizer, not tokenizer.tokenize
    explainer = shap.Explainer(predict, masker, algorithm="partition")

    # Pick test samples
    sample_df = df.sample(10, random_state=0)
    test_samples = sample_df["RequirementText"].tolist()
    true_labels = sample_df["label"].tolist()

    # Run SHAP
    shap_values = explainer(test_samples)

    # Collect results into table
    table_rows = []
    preds = (predict(test_samples) > 0.5).astype(int)

    for i, text in enumerate(test_samples):
        token_scores = list(zip(shap_values.data[i], shap_values.values[i]))
        token_scores_sorted = sorted(token_scores, key=lambda x: abs(x[1]), reverse=True)

        positive_tokens = [tok for tok, score in token_scores_sorted if score > 0][:5]
        negative_tokens = [tok for tok, score in token_scores_sorted if score < 0][:5]

        if preds[i] == 1:  # Functional
            rationale = (
                f"Predicted **Functional** due to terms like {', '.join(positive_tokens)} "
                f"which emphasize concrete actions."
            )
            if negative_tokens:
                rationale += f" Less importance was given to terms like {', '.join(negative_tokens)}."
        else:  # Non-Functional
            rationale = (
                f"Predicted **Non-Functional** due to quality-related words like {', '.join(positive_tokens)}, "
                f"highlighting performance or attributes instead of actions."
            )
            if negative_tokens:
                rationale += f" Some action-related words like {', '.join(negative_tokens)} were down-weighted."

        table_rows.append({
            "Requirement": text,
            "Actual Class": "Functional" if true_labels[i] == 1 else "Non-Functional",
            "Predicted Class": "Functional" if preds[i] == 1 else "Non-Functional",
            "Explanation": rationale
        })

    shap_table = pd.DataFrame(table_rows)
    display(shap_table)
    shap_table.to_csv("shap_explanations_final.csv", index=False)

except Exception as e:
    print(f"⚠️ SHAP failed on final model: {e}")


PartitionExplainer explainer: 11it [00:16,  4.01s/it]                        


,Requirement,Actual Class,Predicted Class,Explanation
0,"The Center shall provide the following link information: link identifier, link name, road number, link type, link type description, start node (see below for node information), end node (see below for node information), direction, length, capacity, speed limit, speed limit truck, number of lanes.",Functional,Functional,"Predicted **Functional** due to terms like Ġdescription, Ġprovide, Ġspeed, see, Ġlink which emphasize concrete actions. Less importance was given to terms like Ġof, Ġnumber, Ġlink, Ġtruck, Ġidentifier."
1,No year designation used in any requirement,Functional,Functional,"Predicted **Functional** due to terms like Ġdesignation, Ġrequirement, No which emphasize concrete actions. Less importance was given to terms like Ġyear, Ġin, Ġany, Ġused."
2,NPAC SMS shall if the old and new service provider timer types match set the subscription version timer type to that timer type.,Non-Functional,Non-Functional,"Predicted **Non-Functional** due to quality-related words like Ġtype, Ġtype, Ġservice, Ġversion, ., highlighting performance or attributes instead of actions. Some action-related words like Ġsubscription, NP, Ġthe, Ġif, Ġtimer were down-weighted."
3,Correction operation to capture and deliver a formatted correction request for transmission by a vessel’s communications system(s) to the UK fisheries administrations’ ERS system to correct previously send data (COR).,Functional,Functional,"Predicted **Functional** due to terms like Correction, Ġcorrection, Ġadministrations, Ġpreviously, Ġtransmission which emphasize concrete actions. Less importance was given to terms like Ġfor, Ġto, Ļ, s, Ġa."
4,The system is configurable to e-mail warnings about multiple failed login attempts from the same user.,Non-Functional,Non-Functional,"Predicted **Non-Functional** due to quality-related words like Ġwarnings, Ġmultiple, Ġfailed, Ġabout, -, highlighting performance or attributes instead of actions. Some action-related words like Ġlogin, mail, Ġ, Ġattempts, Ġuser were down-weighted."
5,"NPAC SMS shall update the Block Failed SP List upon completion of the Activation broadcast, and a response from ALL EDR and non-EDR Local SMSs, or retries are exhausted, as defined in RR3-138.1, and RR3-138.2. (Previously B-275)",Functional,Functional,"Predicted **Functional** due to terms like Previously, Ġ(, Ġresponse, Ġfrom, Ġcompletion which emphasize concrete actions. Less importance was given to terms like ĠRR, Ġand, Ġnon, ĠSMS, NP."
6,NPAC SMS shall provide NPAC SMS personnel with the functionality to re-send modify active Subscription Version requests to all failed Local SMSs.,Functional,Functional,"Predicted **Functional** due to terms like Ġfunctionality, Ġmodify, Ġprovide, ĠSMS, ĠVersion which emphasize concrete actions. Less importance was given to terms like Ġall, Ġfailed, NP, s, Ġto."
7,Customer‟s sales history must be synchronized to QuickBooks POS and roll up in a similar fashion as donation statistics.,Functional,Functional,"Predicted **Functional** due to terms like Ġsimilar, Ġsynchronized, Ġstatistics, Books, Ġhistory which emphasize concrete actions. Less importance was given to terms like Ġas, Ġin, Ġa, Ġsales, Ġroll."
8,"The Center shall support the following status information about each Railroad Crossing: Network identifier (owner of railroad Crossing), Link Identifier, Rail Crossing Identifier, Rail Crossing Name, Location (latitude/longitude), Status, Rail Type, Estimated Time for Train to Clear of Intersection, Estimated Minutes to Train Arrival, Rail closing signal type.",Non-Functional,Non-Functional,"Predicted **Non-Functional** due to quality-related words like Ġinformation, ),, The, Ġabout, :, highlighting performance or attributes instead of actions. Some action-related words like Ġof, ifier, ĠArri, ĠTime, Ġidentifier were down-weighted."
9,NPAC SMS shall default the SOA NPA-NXX-X Indicator to FALSE. (Previously NC-3),Functional,Functional,"Predicted **Functio